# roma-safe-cctv — ingrandimento sulle mani (D44), addestramento su Colab

Addestra il classificatore **niente / pistola / coltello / fucile** sui ritagli intorno ai polsi (vedi `armi/ingrandimento_mani` in `src/sicurezza/visione.py`).
I ritagli sono già pronti nel dataset Kaggle privato `marcozambrella/roma-safe-cctv-ritagli-mani`.

**Prima di partire:** *Runtime → Cambia tipo di runtime → GPU T4*. Poi *Runtime → Esegui tutto*.
Ti verrà chiesto di caricare il tuo `kaggle.json` (Kaggle → Settings → API → Create New Token): resta solo in questa sessione.
Alla fine i pesi vengono caricati nel dataset Kaggle privato `marcozambrella/roma-safe-cctv-mani-modello` (e scaricati anche sul tuo computer).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone -q --depth 1 https://github.com/marcozambrella/roma-safe-cctv /content/repo
!pip install -q "ultralytics>=8.4" kaggle
import sys; sys.path.insert(0, "/content/repo/src")

Carica il tuo `kaggle.json`:

In [ ]:
import os, shutil
from google.colab import files
caricati = files.upload()
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.move(next(iter(caricati)), "/root/.kaggle/kaggle.json")
os.chmod("/root/.kaggle/kaggle.json", 0o600)
!kaggle datasets list --mine | head -5

In [ ]:
!kaggle datasets download marcozambrella/roma-safe-cctv-ritagli-mani -p /content --unzip -q
!cd /content && (ls ritagli_mani.zip >/dev/null 2>&1 && unzip -q -o ritagli_mani.zip -d /content) ; ls /content/ritagli_mani
!for s in train val test; do for c in niente pistol knife rifle; do echo "$s $c $(ls /content/ritagli_mani/$s/$c 2>/dev/null | wc -l)"; done; done

Addestramento (~30-60 minuti su T4):

In [ ]:
!cd /content/repo && PYTHONPATH=src python -m sicurezza.visione addestra-mani --dati /content/ritagli_mani --uscita /content/runs --dispositivo 0 --epoche 30

Valutazione sul test dei ritagli:

In [ ]:
from ultralytics import YOLO
modello = YOLO("/content/runs/yolo26s_mani/weights/best.pt")
metriche = modello.val(data="/content/ritagli_mani", split="test", imgsz=224)
print("accuratezza top-1 sul test:", metriche.top1)

Carica i pesi su Kaggle (dataset privato) e scaricali anche qui:

In [ ]:
import json, shutil, subprocess
os.makedirs("/content/mani_modello", exist_ok=True)
shutil.copy("/content/runs/yolo26s_mani/weights/best.pt", "/content/mani_modello/mani_best.pt")
shutil.copy("/content/runs/yolo26s_mani/results.csv", "/content/mani_modello/results.csv")
json.dump({"title": "roma-safe-cctv mani modello", "id": "marcozambrella/roma-safe-cctv-mani-modello",
           "licenses": [{"name": "other"}]}, open("/content/mani_modello/dataset-metadata.json", "w"))
esiste = subprocess.run(["kaggle", "datasets", "status", "marcozambrella/roma-safe-cctv-mani-modello"],
                        capture_output=True, text=True).returncode == 0
comando = (["kaggle", "datasets", "version", "-p", "/content/mani_modello", "-m", "nuovo addestramento"] if esiste
           else ["kaggle", "datasets", "create", "-p", "/content/mani_modello"])
print(subprocess.run(comando, capture_output=True, text=True).stdout)
files.download("/content/mani_modello/mani_best.pt")